# Business Analysis — Credit Risk & Account Balance

**Team:** Equip_32 — Finance & Credit Risk Analysts  
**Dataset:** Bank Marketing (UCI-inspired), cleaned export `bank_dataset_2026-06-29.csv`  
**Prerequisite notebook:** `EDA_bank_2026-06-29.ipynb` (data understanding, cleaning, EDA)

---

## Business Question

> **To what extent are customers with lower account balances at greater risk of credit default, and what credit policies should the bank adopt to mitigate this risk?**

### Hypothesis

| | Statement |
|---|---|
| **H₀** | Account balance and credit default are independent. |
| **H₁** | Lower balances are associated with a higher probability of credit default. |

### Analytical approach

This notebook moves from **exploratory analysis** to **hypothesis-driven business analysis**:

1. Measure overall default risk
2. Compare balance distributions between defaulters and non-defaulters
3. Segment customers by balance level and quantify default rates
4. Cross-balance risk with loan, housing, age, and job
5. Validate findings with statistical tests
6. Translate evidence into credit policy recommendations

## 1. Setup & Data Loading

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Visual style (consistent with EDA notebook)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 13
plt.rcParams["xtick.labelsize"] = 11
plt.rcParams["ytick.labelsize"] = 11

DATA_PATH = Path("../Data/bank_dataset_2026-06-29.csv")
RESULTS_DIR = Path("../Results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
print(f"Raw records: {len(df_raw):,}")
df_raw.head()

In [ ]:
# Focus on variables relevant to the business question
ANALYSIS_COLS = [
    "id", "balance", "default", "loan", "housing", "age", "job", "deposit"
]

df = df_raw[ANALYSIS_COLS].copy()

# Binary target: 1 = credit default, 0 = no default
df["default_flag"] = (df["default"] == "yes").astype(int)

# Drop rows with missing values in key analysis fields
df = df.dropna(subset=["balance", "default", "age"]).copy()

print(f"Analysis sample: {len(df):,} customers")
print(f"Overall default rate: {df['default_flag'].mean():.2%}")
print(f"Number of defaults: {df['default_flag'].sum()}")

## 2. Baseline Default Rate

Before segmenting by balance, we establish the **portfolio baseline**: the proportion of customers who have defaulted on credit.

In [ ]:
default_summary = (
    df["default"]
    .value_counts()
    .rename_axis("default")
    .reset_index(name="customers")
)
default_summary["default_rate"] = default_summary["customers"] / default_summary["customers"].sum()
default_summary

In [ ]:
fig, ax = plt.subplots()
colors = ["#2ecc71", "#e74c3c"]
bars = ax.bar(default_summary["default"], default_summary["default_rate"] * 100, color=colors)
ax.set_title("Portfolio Default Rate")
ax.set_xlabel("Credit Default")
ax.set_ylabel("Percentage (%)")
ax.set_ylim(0, max(default_summary["default_rate"] * 100) * 1.3)

for bar, rate in zip(bars, default_summary["default_rate"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
            f"{rate:.2%}", ha="center", va="bottom", fontweight="bold")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "default_rate_overview.png", dpi=150, bbox_inches="tight")
plt.show()

**Interpretation:** Default is a rare event (~1.5% of the portfolio). This class imbalance is expected in credit risk datasets and reinforces the need to analyse **rates within segments**, not raw counts alone.

## 3. Balance vs Credit Default

We compare account balance between customers who defaulted and those who did not, using both **mean** (sensitive to outliers) and **median** (robust).

In [ ]:
balance_by_default = (
    df.groupby("default")["balance"]
    .agg(count="count", mean="mean", median="median", std="std")
    .round(2)
)
balance_by_default

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

metrics = {"mean": "Mean Balance (€)", "median": "Median Balance (€)"}
palette = {"no": "#3498db", "yes": "#e74c3c"}

for ax, (metric, ylabel) in zip(axes, metrics.items()):
    plot_data = balance_by_default[metric]
    sns.barplot(x=plot_data.index, y=plot_data.values, hue=plot_data.index,
                palette=palette, legend=False, ax=ax)
    ax.set_title(f"{ylabel} by Default Status")
    ax.set_xlabel("Credit Default")
    ax.set_ylabel(ylabel)
    for i, val in enumerate(plot_data.values):
        ax.text(i, val + (50 if val >= 0 else -200), f"€{val:,.0f}", ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "balance_mean_median_by_default.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x="default", y="balance", hue="default",
            palette={"no": "#3498db", "yes": "#e74c3c"}, legend=False)
plt.title("Account Balance Distribution by Credit Default")
plt.xlabel("Credit Default")
plt.ylabel("Balance (€)")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "balance_boxplot_by_default.png", dpi=150, bbox_inches="tight")
plt.show()

**Key finding:** Customers who default have a **negative mean balance (~€−74)** and a **median of €0**, while non-defaulters average **~€1,560** (median **€570**). The boxplot confirms that defaulters cluster at the bottom of the balance distribution.

## 4. Default Rate by Balance Segment

Continuous balance is hard to act on operationally. We segment customers into **balance tiers** and compute the default rate for each group.

Two segmentation schemes:
- **Quartiles** — data-driven, equal-sized groups
- **Business tiers** — interpretable thresholds for policy design

In [ ]:
def default_rate_table(data, group_col):
    """Compute default rate summary for a grouping column."""
    summary = (
        data.groupby(group_col, observed=True)
        .agg(customers=("default_flag", "count"),
             defaults=("default_flag", "sum"),
             default_rate=("default_flag", "mean"))
        .reset_index()
    )
    summary["default_rate_pct"] = (summary["default_rate"] * 100).round(2)
    return summary

In [ ]:
# Quartile segmentation
df["balance_quartile"] = pd.qcut(
    df["balance"], q=4,
    labels=["Q1 (lowest)", "Q2", "Q3", "Q4 (highest)"]
)

quartile_rates = default_rate_table(df, "balance_quartile")
quartile_rates

In [ ]:
# Business-tier segmentation
balance_bins = [-np.inf, 0, 500, 1500, 3000, np.inf]
balance_labels = ["Negative (≤ €0)", "€0–500", "€500–1,500", "€1,500–3,000", "> €3,000"]

df["balance_tier"] = pd.cut(df["balance"], bins=balance_bins, labels=balance_labels)

tier_rates = default_rate_table(df, "balance_tier")
tier_rates

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, table, title in [
    (axes[0], quartile_rates, "Default Rate by Balance Quartile"),
    (axes[1], tier_rates, "Default Rate by Balance Tier"),
]:
    group_col = table.columns[0]
    bars = ax.bar(table[group_col].astype(str), table["default_rate_pct"],
                  color=sns.color_palette("RdYlGn_r", len(table)))
    ax.set_title(title)
    ax.set_xlabel("Balance Segment")
    ax.set_ylabel("Default Rate (%)")
    ax.tick_params(axis="x", rotation=30)
    for bar, rate in zip(bars, table["default_rate_pct"]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
                f"{rate:.2f}%", ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "default_rate_by_balance_segment.png", dpi=150, bbox_inches="tight")
plt.show()

### Star table — default rate by balance quartile

| Balance quartile | Customers | Defaults | Default rate |
|---|---:|---:|---:|
| Q1 (lowest) | 2,669 | 127 | **4.76%** |
| Q2 | 2,649 | 16 | 0.60% |
| Q3 | 2,658 | 10 | 0.38% |
| Q4 (highest) | 2,657 | 2 | **0.08%** |

Customers in the **lowest balance quartile** have a default rate approximately **63× higher** than those in the highest quartile.

## 5. Risk Interactions — Balance × Other Variables

Balance alone does not explain all default risk. We cross low vs high balance with **loan**, **housing**, **age**, and **job** to identify compounded risk profiles.

In [ ]:
df["balance_level"] = np.where(df["balance"] <= 500, "Low (≤ €500)", "High (> €500)")

def cross_default_rate(data, row_col, col_col):
    """Pivot table of default rates for two categorical dimensions."""
    grouped = (
        data.groupby([row_col, col_col], observed=True)
        .agg(default_rate=("default_flag", "mean"), n=("default_flag", "count"))
        .reset_index()
    )
    pivot_rate = grouped.pivot(index=row_col, columns=col_col, values="default_rate")
    pivot_n = grouped.pivot(index=row_col, columns=col_col, values="n")
    return pivot_rate, pivot_n

In [ ]:
# Balance × Personal loan
loan_rate, loan_n = cross_default_rate(df, "balance_level", "loan")
print("Default rate — Balance level × Personal loan")
display((loan_rate * 100).round(2).astype(str) + "%")
print("\nSample sizes:")
display(loan_n.astype(int))

In [ ]:
# Balance × Housing loan
housing_rate, housing_n = cross_default_rate(df, "balance_level", "housing")
print("Default rate — Balance level × Housing loan")
display((housing_rate * 100).round(2).astype(str) + "%")

In [ ]:
# Balance × Age group
df["age_group"] = pd.cut(
    df["age"], bins=[0, 30, 45, 60, 100],
    labels=["≤ 30", "31–45", "46–60", "> 60"]
)

age_rate, age_n = cross_default_rate(df, "balance_level", "age_group")
print("Default rate — Balance level × Age group")
display((age_rate * 100).round(2).astype(str) + "%")

In [ ]:
# Default rate by job (low-balance customers only, min 5 observations)
low_balance = df[df["balance"] <= 500]

job_risk = (
    low_balance.groupby("job")
    .agg(customers=("default_flag", "count"),
         defaults=("default_flag", "sum"),
         default_rate=("default_flag", "mean"))
    .query("customers >= 5")
    .sort_values("default_rate", ascending=False)
)
job_risk["default_rate_pct"] = (job_risk["default_rate"] * 100).round(2)
job_risk

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
plot_jobs = job_risk.head(8).reset_index()
sns.barplot(data=plot_jobs, x="default_rate_pct", y="job", hue="job",
            palette="Reds_r", legend=False, ax=ax)
ax.set_title("Default Rate by Job (Low-Balance Customers ≤ €500)")
ax.set_xlabel("Default Rate (%)")
ax.set_ylabel("Job Category")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "default_rate_job_low_balance.png", dpi=150, bbox_inches="tight")
plt.show()

**Interaction insights:**

- **Low balance + personal loan** yields the highest default rate (~6.1%), roughly **3× the rate** for high-balance customers with a loan.
- Housing loan status has a smaller effect within each balance band.
- Among low-balance customers, **entrepreneurs, unemployed, and blue-collar workers** show the highest default rates — useful for risk-based pricing and manual review triggers.

## 6. Statistical Validation

Visual patterns must be supported by formal tests before driving policy decisions.

In [ ]:
# Mann-Whitney U — balance distributions differ between defaulters and non-defaulters?
# Appropriate because balance is non-normal and contains extreme outliers.
balance_default = df.loc[df["default"] == "yes", "balance"]
balance_no_default = df.loc[df["default"] == "no", "balance"]

u_stat, u_pvalue = stats.mannwhitneyu(balance_default, balance_no_default, alternative="two-sided")

print("Mann-Whitney U test — balance by default status")
print(f"  U statistic : {u_stat:,.1f}")
print(f"  p-value     : {u_pvalue:.2e}")
print(f"  Conclusion  : {'Reject H₀ — distributions are significantly different' if u_pvalue < 0.05 else 'Fail to reject H₀'}")

In [ ]:
# Chi-square — balance quartile associated with default?
contingency = pd.crosstab(df["balance_quartile"], df["default_flag"])
chi2, chi_p, dof, expected = stats.chi2_contingency(contingency)

print("Chi-square test — balance quartile vs default")
print(f"  χ² statistic : {chi2:.2f}")
print(f"  df           : {dof}")
print(f"  p-value      : {chi_p:.2e}")
print(f"  Conclusion   : {'Reject H₀ — balance segment and default are associated' if chi_p < 0.05 else 'Fail to reject H₀'}")
print("\nContingency table:")
display(contingency)

In [ ]:
def odds_ratio_and_rr(group_a, group_b, label_a="Group A", label_b="Group B"):
    """Compute odds ratio and relative risk between two binary-outcome groups."""
    a = group_a["default_flag"].sum()
    b = len(group_a) - a
    c = group_b["default_flag"].sum()
    d = len(group_b) - c

    odds_a = a / b if b > 0 else np.inf
    odds_b = c / d if d > 0 else np.inf
    or_value = odds_a / odds_b if odds_b > 0 else np.inf

    rate_a = a / len(group_a)
    rate_b = c / len(group_b)
    rr_value = rate_a / rate_b if rate_b > 0 else np.inf

    return pd.DataFrame({
        "comparison": [f"{label_a} vs {label_b}"],
        f"{label_a}_default_rate": [f"{rate_a:.2%}"],
        f"{label_b}_default_rate": [f"{rate_b:.2%}"],
        "odds_ratio": [round(or_value, 2)],
        "relative_risk": [round(rr_value, 2)],
    })

In [ ]:
# Negative/zero balance vs positive balance
non_positive = df[df["balance"] <= 0]
positive = df[df["balance"] > 0]

risk_comparison = odds_ratio_and_rr(
    non_positive, positive,
    label_a="Balance ≤ €0", label_b="Balance > €0"
)
risk_comparison

In [ ]:
# Lowest vs highest quartile
q1 = df[df["balance_quartile"] == "Q1 (lowest)"]
q4 = df[df["balance_quartile"] == "Q4 (highest)"]

quartile_comparison = odds_ratio_and_rr(
    q1, q4,
    label_a="Q1 (lowest balance)", label_b="Q4 (highest balance)"
)
quartile_comparison

**Statistical summary:**

- **Mann-Whitney U** (p ≈ 5.2 × 10⁻⁵²): balance distributions of defaulters and non-defaulters are significantly different.
- **Chi-square** (p ≈ 7.5 × 10⁻⁵⁹): balance quartile and default status are strongly associated.
- **Odds ratio** (balance ≤ €0 vs > €0): customers with non-positive balance are **~12× more likely** to default.
- **Relative risk** (Q1 vs Q4): lowest-quartile customers carry **~63× the default rate** of the highest quartile.

## 7. Credit Policy Recommendations

Based on the quantitative evidence above, we propose the following adjustments to credit granting policies:

| # | Policy | Trigger condition | Recommended action | Rationale |
|---|---|---|---|---|
| **1** | **Mandatory manual review** | Balance ≤ €0 | Block automated approval; escalate to credit analyst | 7.0% default rate — ~11× portfolio average |
| **2** | **Reduced credit limit** | Balance €0–500 | Cap new credit at ≤ 50% of standard limit | 2.8% default rate; 2× baseline risk |
| **3** | **Enhanced scoring weight** | All applicants | Increase balance weight in internal credit score | Balance is the strongest single predictor in this dataset |
| **4** | **Compound-risk flag** | Balance ≤ €500 **and** personal loan = yes | Require additional guarantees or co-signer | ~6.1% default rate — highest observed segment |
| **5** | **Occupation-based review** | Balance ≤ €500 **and** job ∈ {entrepreneur, unemployed, blue-collar} | Pre-approval financial health check | These jobs show 4–6% default within low-balance segment |
| **6** | **Preventive outreach** | Balance in Q1 quartile, no current default | Proactive financial counselling before new credit offers | Early intervention before delinquency |
| **7** | **Standard/fast-track approval** | Balance > €3,000, no loan | Maintain current streamlined process | 0.07% default rate — minimal risk |

## 8. Business Conclusions

### Direct answer to the business question

**1. To what extent are lower-balance customers at greater default risk?**

The relationship is **strong, quantifiable, and statistically significant**. Customers with the lowest balances (Q1 quartile) default at **4.76%**, compared with **0.08%** in the highest quartile — a **63-fold difference**. Non-positive balances (≤ €0) show a **7.0% default rate**, nearly **11 times** the portfolio baseline of 1.46%.

**2. How should credit policies be adjusted?**

The bank should treat account balance as a **primary risk signal**, not a secondary variable:

- Introduce **tiered approval workflows** aligned with balance segments (see Policy table above).
- Apply **stricter conditions** when low balance combines with personal loans or high-risk occupations.
- Embed balance quartile as a **mandatory input** in the credit scoring model.
- Reserve **fast-track approval** for customers with balance > €3,000 and clean credit history.

### Limitations & next steps

- **Class imbalance:** Only 155 defaults in 10,633 records — segment rates for rare groups (e.g. Q4) have wide confidence intervals.
- **Observational data:** This analysis shows association, not causation. A customer’s low balance may reflect prior financial distress rather than cause future default.
- **Recommended follow-up:** Build a multivariate logistic regression or decision-tree model including balance, loan, job, and age to estimate adjusted risk and validate policy thresholds.

---

*Notebook prepared by Equip_32 — Finance & Credit Risk Analysts. Data source: UCI Bank Marketing dataset (adapted).*